# Complete NDVI SQS Workflow from Scratch  
## Updated for your project folder structure

project structure:

```text
final-project-greenness-infrastructure-happiness/
  boundary_data/
  CGSS2018/
  code/
  manifests/
  ndvi_output/
  satellite_data/
  tasks_csv/
```

This notebook will:

1. create/reuse the S3 bucket `final-project-ndvi`;
2. upload code, manifests, boundaries, task CSV, and cleaned CGSS data;
3. create/reuse the SQS queue `ndvi-province-tasks`;
4. enqueue one province-level task per row of `tasks_csv/province_tasks.csv`;
5. create/reuse an EC2 key pair;
6. launch EC2 workers;
7. prepare or repair each worker;
8. configure NASA temporary credentials on each worker;
9. start workers in `tmux`;
10. monitor SQS and download final NDVI outputs.

The notebook is designed for AWS Academy Learner Lab. Make sure your lab is running and your local AWS credentials are active before starting.


## 0. Install/import local notebook dependencies


In [1]:
import sys
import subprocess
import importlib.util

required = ["boto3", "botocore", "pandas"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Notebook dependencies are already installed.")


Notebook dependencies are already installed.


In [2]:
from pathlib import Path
import os
import json
import time
import shlex
import getpass
import subprocess
import platform
import pandas as pd

import boto3
from botocore.exceptions import ClientError


## 1. Detect project root and set local paths


In [3]:
def detect_project_root() -> Path:
    """
    Find the project root by looking upward from the current working directory.
    The root should contain these folders:
      code, boundary_data, manifests, tasks_csv
    """
    start = Path.cwd().resolve()
    required_dirs = ["code", "boundary_data", "manifests", "tasks_csv"]

    for candidate in [start, *start.parents]:
        if all((candidate / d).exists() for d in required_dirs):
            return candidate

    raise FileNotFoundError(
        "Could not detect project root. Open this notebook from the project root, "
        "or from a subfolder under the root containing code/, boundary_data/, manifests/, and tasks_csv/."
    )

PROJECT_ROOT = detect_project_root()
os.chdir(PROJECT_ROOT)

CODE_DIR = PROJECT_ROOT / "code"
BOUNDARY_DIR = PROJECT_ROOT / "boundary_data"
MANIFEST_DIR = PROJECT_ROOT / "manifests"
TASKS_DIR = PROJECT_ROOT / "tasks_csv"
OUTPUT_DIR = PROJECT_ROOT / "ndvi_output"
CGSS_DIR = PROJECT_ROOT / "CGSS2018"
SATELLITE_DIR = PROJECT_ROOT / "satellite_data"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CODE_DIR:", CODE_DIR)
print("BOUNDARY_DIR:", BOUNDARY_DIR)
print("MANIFEST_DIR:", MANIFEST_DIR)
print("TASKS_DIR:", TASKS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CGSS_DIR:", CGSS_DIR)


PROJECT_ROOT: C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness
CODE_DIR: C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\code
BOUNDARY_DIR: C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\boundary_data
MANIFEST_DIR: C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\manifests
TASKS_DIR: C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\tasks_csv
OUTPUT_DIR: C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\ndvi_output
CGSS_DIR: C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\CGSS2018


In [4]:
# Main local files based on your screenshot/project structure.
LAUNCHER_LOCAL = CODE_DIR / "launch_ndvi_workers.py"
ENQUEUE_LOCAL = CODE_DIR / "enqueue_ndvi_tasks.py"
WORKER_LOCAL = CODE_DIR / "worker_poll_sqs_ndvi_old.py"
COMPUTE_LOCAL = CODE_DIR / "compute_hls_ndvi_by_province.py"
CLOUD_COMPUTE_LOCAL = CODE_DIR / "compute_hls_ndvi_by_province_cloud_coverage.py"

BOUNDARY_LOCAL = BOUNDARY_DIR / "geoBoundaries-CHN-ADM1.geojson"
CGSS_CLEAN_LOCAL = CGSS_DIR / "CGSS2018_clean_main.csv"
TASKS_CSV_LOCAL = TASKS_DIR / "province_tasks.csv"

MANIFEST_FILES = sorted(MANIFEST_DIR.glob("*.txt"))

required_files = {
    "launcher": LAUNCHER_LOCAL,
    "enqueue script": ENQUEUE_LOCAL,
    "SQS worker script": WORKER_LOCAL,
    "compute script": COMPUTE_LOCAL,
    "ADM1 boundary": BOUNDARY_LOCAL,
    "province task CSV": TASKS_CSV_LOCAL,
}

missing = [f"{label}: {path}" for label, path in required_files.items() if not path.exists()]
if missing:
    print("Missing required files:")
    for x in missing:
        print(" -", x)
    raise FileNotFoundError("Fix the missing files before continuing.")

if not MANIFEST_FILES:
    raise FileNotFoundError(f"No manifest .txt files found under {MANIFEST_DIR}")

print("Required files found.")
print("Number of manifest files:", len(MANIFEST_FILES))
print("Cloud compute script exists:", CLOUD_COMPUTE_LOCAL.exists())
print("Cleaned CGSS exists:", CGSS_CLEAN_LOCAL.exists())


Required files found.
Number of manifest files: 30
Cloud compute script exists: False
Cleaned CGSS exists: True


## 2. AWS configuration


In [6]:
# ---------- AWS resource names ----------
PROJECT_BUCKET = "final-project-ndvi"
QUEUE_NAME = "ndvi-province-tasks"

# S3 bucket and EC2 workers are in us-west-2 because NASA HLS S3 is also us-west-2.
BUCKET_REGION = "us-west-2"
EC2_REGION = "us-west-2"

# Your SQS queue is in us-east-1.
SQS_REGION = "us-east-1"

# ---------- S3 layout ----------
SCRIPTS_PREFIX = "scripts"
BOUNDARIES_PREFIX = "boundaries"
MANIFEST_PREFIX = "manifests"
TASKS_PREFIX = "tasks"
CGSS_PREFIX = "cgss"
OUTPUT_PREFIX = "outputs"

WORKER_SCRIPT_KEY = f"{SCRIPTS_PREFIX}/worker_poll_sqs_ndvi_old.py"
PROCESSING_SCRIPT_KEY = f"{SCRIPTS_PREFIX}/compute_hls_ndvi_by_province.py"
CLOUD_PROCESSING_SCRIPT_KEY = f"{SCRIPTS_PREFIX}/compute_hls_ndvi_by_province_cloud_coverage.py"
BOUNDARY_KEY = f"{BOUNDARIES_PREFIX}/geoBoundaries-CHN-ADM1.geojson"
TASKS_CSV_KEY = f"{TASKS_PREFIX}/province_tasks.csv"

# ---------- EC2 worker settings ----------
NUM_WORKERS = 2
INSTANCE_TYPE = "m5.large"
VOLUME_SIZE_GB = 100
SWAP_GB = 32

IAM_INSTANCE_PROFILE = "LabInstanceProfile"
SECURITY_GROUP_NAME = "ndvi-worker-sg"
KEY_NAME = "ndvi-key"
PEM_PATH = Path.home() / ".ssh" / f"{KEY_NAME}.pem"

# Worker polling settings.
YEAR = 2018
VISIBILITY_TIMEOUT = 21600
MAX_EMPTY_POLLS = 5

# Local output files/folders.
LAUNCHED_CSV = CODE_DIR / "launched_ndvi_workers.csv"
LOCAL_S3_OUTPUT_DIR = OUTPUT_DIR / "s3_outputs"
FINAL_ANNUAL_NDVI_CSV = OUTPUT_DIR / "province_annual_ndvi_2018_combined.csv"
FINAL_DAILY_NDVI_CSV = OUTPUT_DIR / "province_daily_ndvi_2018_combined.csv"

print("Project bucket:", PROJECT_BUCKET)
print("Queue name:", QUEUE_NAME)
print("PEM path:", PEM_PATH)
print("Launched worker CSV:", LAUNCHED_CSV)


Project bucket: final-project-ndvi
Queue name: ndvi-province-tasks
PEM path: C:\Users\26540\.ssh\ndvi-key.pem
Launched worker CSV: C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\code\launched_ndvi_workers.csv


## 3. Helper functions


In [7]:
def run_local(cmd, *, cwd=None, check=True, capture=True, input_text=None, timeout=None):
    """
    Run a local command. Pass cmd as a list, not a string, whenever possible.
    """
    print("\n$", " ".join(shlex.quote(str(x)) for x in cmd))
    result = subprocess.run(
        cmd,
        cwd=cwd,
        input=input_text,
        text=True,
        capture_output=capture,
        timeout=timeout,
    )
    if capture:
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, result.stdout, result.stderr)
    return result


def client(service, region):
    return boto3.client(service, region_name=region)


def resource(service, region):
    return boto3.resource(service, region_name=region)


def s3_uri(key: str) -> str:
    return f"s3://{PROJECT_BUCKET}/{key}"


def upload_file(local_path: Path, key: str, s3_client=None):
    """
    Upload one local file to the project bucket.
    """
    if s3_client is None:
        s3_client = client("s3", BUCKET_REGION)
    local_path = Path(local_path)
    if not local_path.exists():
        raise FileNotFoundError(local_path)
    print(f"Uploading {local_path.relative_to(PROJECT_ROOT)} -> {s3_uri(key)}")
    s3_client.upload_file(str(local_path), PROJECT_BUCKET, key)


def queue_counts(queue_url: str):
    sqs = client("sqs", SQS_REGION)
    response = sqs.get_queue_attributes(
        QueueUrl=queue_url,
        AttributeNames=[
            "ApproximateNumberOfMessages",
            "ApproximateNumberOfMessagesNotVisible",
            "ApproximateNumberOfMessagesDelayed",
        ],
    )
    return response["Attributes"]


## 4. Check AWS credentials


In [8]:
sts = client("sts", BUCKET_REGION)
identity = sts.get_caller_identity()

print("AWS identity:")
print(json.dumps(identity, indent=2))


AWS identity:
{
  "UserId": "AROA47CRYUBNG6DOHZ7UZ:user5003845=tingruizhang@uchicago.edu",
  "Account": "891377197146",
  "Arn": "arn:aws:sts::891377197146:assumed-role/voclabs/user5003845=tingruizhang@uchicago.edu",
  "ResponseMetadata": {
    "RequestId": "c2d98f16-0072-4566-9fa7-ebc2ade14dbd",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "c2d98f16-0072-4566-9fa7-ebc2ade14dbd",
      "x-amz-sts-extended-request-id": "MTp1cy13ZXN0LTI6UzoxNzc5ODEzMjkxNjE2OlI6b2o0bTFtcmU=",
      "content-type": "text/xml",
      "content-length": "488",
      "date": "Tue, 26 May 2026 16:34:51 GMT"
    },
    "RetryAttempts": 0
  }
}


## 5. Create or reuse S3 bucket `final-project-ndvi`


In [9]:
s3 = client("s3", BUCKET_REGION)

def ensure_bucket(bucket_name: str, region: str):
    try:
        s3.head_bucket(Bucket=bucket_name)
        print(f"Bucket already exists and is accessible: s3://{bucket_name}")
        return
    except ClientError as e:
        code = e.response.get("Error", {}).get("Code")
        print("head_bucket result:", code)

    print(f"Creating bucket: s3://{bucket_name} in {region}")
    if region == "us-east-1":
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region},
        )
    print("Bucket created.")

ensure_bucket(PROJECT_BUCKET, BUCKET_REGION)


Bucket already exists and is accessible: s3://final-project-ndvi


## 6. Upload local code, boundaries, manifests, task CSV, and cleaned CGSS data


In [10]:
# Upload scripts expected by the launcher and workers.
upload_file(LAUNCHER_LOCAL, f"{SCRIPTS_PREFIX}/launch_ndvi_workers.py", s3)
upload_file(ENQUEUE_LOCAL, f"{SCRIPTS_PREFIX}/enqueue_ndvi_tasks.py", s3)
upload_file(WORKER_LOCAL, WORKER_SCRIPT_KEY, s3)
upload_file(COMPUTE_LOCAL, PROCESSING_SCRIPT_KEY, s3)

# Optional cloud-coverage compute script. The launcher can tolerate this being absent, but upload it if present.
if CLOUD_COMPUTE_LOCAL.exists():
    upload_file(CLOUD_COMPUTE_LOCAL, CLOUD_PROCESSING_SCRIPT_KEY, s3)
else:
    print("Optional cloud compute script not found; skipping upload:", CLOUD_COMPUTE_LOCAL)

# Upload all boundary GeoJSON files, including ADM1.
for p in sorted(BOUNDARY_DIR.glob("*.geojson")):
    upload_file(p, f"{BOUNDARIES_PREFIX}/{p.name}", s3)

# Upload all province manifest files.
for p in MANIFEST_FILES:
    upload_file(p, f"{MANIFEST_PREFIX}/{p.name}", s3)

# Upload task CSV.
upload_file(TASKS_CSV_LOCAL, TASKS_CSV_KEY, s3)

# Upload cleaned CGSS if it exists.
if CGSS_CLEAN_LOCAL.exists():
    upload_file(CGSS_CLEAN_LOCAL, f"{CGSS_PREFIX}/{CGSS_CLEAN_LOCAL.name}", s3)
else:
    print("Cleaned CGSS file not found; skipping:", CGSS_CLEAN_LOCAL)

print("Upload step complete.")


Uploading code\launch_ndvi_workers.py -> s3://final-project-ndvi/scripts/launch_ndvi_workers.py
Uploading code\enqueue_ndvi_tasks.py -> s3://final-project-ndvi/scripts/enqueue_ndvi_tasks.py
Uploading code\worker_poll_sqs_ndvi_old.py -> s3://final-project-ndvi/scripts/worker_poll_sqs_ndvi_old.py
Uploading code\compute_hls_ndvi_by_province.py -> s3://final-project-ndvi/scripts/compute_hls_ndvi_by_province.py
Optional cloud compute script not found; skipping upload: C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\code\compute_hls_ndvi_by_province_cloud_coverage.py
Uploading boundary_data\geoBoundaries-CHN-ADM0.geojson -> s3://final-project-ndvi/boundaries/geoBoundaries-CHN-ADM0.geojson
Uploading boundary_data\geoBoundaries-CHN-ADM1.geojson -> s3://final-project-ndvi/boundaries/geoBoundaries-CHN-ADM1.geojson
Uploading manifests\anhui_ndvi_s3.txt -> s3://final-project-ndvi/manifests/anhui_ndvi_s3.txt
Uploading manifests\beijing_ndvi_s3.txt -> s3://final-p

## 7. Create or reuse SQS queue `ndvi-province-tasks`


In [12]:
from botocore.exceptions import ClientError

sqs = client("sqs", SQS_REGION)

queue_attrs = {
    "VisibilityTimeout": str(VISIBILITY_TIMEOUT),
    "ReceiveMessageWaitTimeSeconds": "20",
}

try:
    response = sqs.create_queue(
        QueueName=QUEUE_NAME,
        Attributes=queue_attrs,
    )
    QUEUE_URL = response["QueueUrl"]
    print("Created queue:")
    print("QUEUE_URL:", QUEUE_URL)

except ClientError as e:
    code = e.response["Error"]["Code"]

    if code in ["QueueAlreadyExists", "QueueNameExists", "AWS.SimpleQueueService.QueueNameExists"]:
        response = sqs.get_queue_url(QueueName=QUEUE_NAME)
        QUEUE_URL = response["QueueUrl"]

        sqs.set_queue_attributes(
            QueueUrl=QUEUE_URL,
            Attributes=queue_attrs,
        )

        print("Queue already exists. Reusing and updating attributes:")
        print("QUEUE_URL:", QUEUE_URL)

    else:
        raise

attrs = sqs.get_queue_attributes(
    QueueUrl=QUEUE_URL,
    AttributeNames=[
        "VisibilityTimeout",
        "ReceiveMessageWaitTimeSeconds",
        "ApproximateNumberOfMessages",
        "ApproximateNumberOfMessagesNotVisible",
    ],
)["Attributes"]

attrs


Queue already exists. Reusing and updating attributes:
QUEUE_URL: https://sqs.us-east-1.amazonaws.com/891377197146/ndvi-province-tasks


{'VisibilityTimeout': '21600',
 'ReceiveMessageWaitTimeSeconds': '20',
 'ApproximateNumberOfMessages': '2',
 'ApproximateNumberOfMessagesNotVisible': '0'}

### Optional: purge old messages before enqueueing

Only run this if you are sure no workers are currently processing messages. SQS may reject repeated purges within 60 seconds.


In [14]:
PURGE_QUEUE_BEFORE_ENQUEUE = True

if PURGE_QUEUE_BEFORE_ENQUEUE:
    print("Purging queue. Do not run this while workers are processing.")
    sqs.purge_queue(QueueUrl=QUEUE_URL)
    print("Purge requested. Waiting 60 seconds...")
    time.sleep(60)
    print(queue_counts(QUEUE_URL))
else:
    print("Skipping purge.")


Purging queue. Do not run this while workers are processing.
Purge requested. Waiting 60 seconds...
{'ApproximateNumberOfMessages': '0', 'ApproximateNumberOfMessagesNotVisible': '0', 'ApproximateNumberOfMessagesDelayed': '0'}


## 8. Validate and enqueue province tasks


In [15]:
tasks = pd.read_csv(TASKS_CSV_LOCAL)
display(tasks.head())
print("Number of rows:", len(tasks))
print("Columns:", list(tasks.columns))

required_cols = {"province", "manifest_file", "output_folder"}
missing_cols = required_cols - set(tasks.columns)
if missing_cols:
    raise ValueError(f"Task CSV is missing required columns: {sorted(missing_cols)}")

# Verify that each manifest in the task CSV exists locally.
missing_manifests = []
for mf in tasks["manifest_file"].dropna().astype(str):
    if not (MANIFEST_DIR / mf).exists():
        missing_manifests.append(mf)

if missing_manifests:
    print("Missing manifest files listed in task CSV:")
    for x in missing_manifests[:20]:
        print(" -", x)
    raise FileNotFoundError("Some task CSV manifest_file entries are missing locally.")
else:
    print("All task CSV manifest files exist locally.")


,province,manifest_file,output_folder
0,Anhui,anhui_ndvi_s3.txt,anhui
1,Beijing,beijing_ndvi_s3.txt,beijing
2,Chongqing,chongqing_ndvi_s3.txt,chongqing
3,Fujian,fujian_ndvi_s3.txt,fujian
4,Gansu,gansu_ndvi_s3.txt,gansu


Number of rows: 29
Columns: ['province', 'manifest_file', 'output_folder']
All task CSV manifest files exist locally.


In [17]:
# Enqueue using your existing enqueue_ndvi_tasks.py script.
# This script creates messages with province, year, manifest_s3, output_s3, boundary_s3, and script_s3.
ENQUEUE_TASKS = True

if ENQUEUE_TASKS:
    run_local([
        sys.executable,
        str(ENQUEUE_LOCAL),
        "--queue-url", QUEUE_URL,
        "--tasks-csv", str(TASKS_CSV_LOCAL),
        "--bucket", PROJECT_BUCKET,
        "--manifest-prefix", MANIFEST_PREFIX,
        "--output-prefix", OUTPUT_PREFIX,
        "--script-key", PROCESSING_SCRIPT_KEY,
        "--boundary-key", BOUNDARY_KEY,
        "--year", str(YEAR),
        "--sqs-region", SQS_REGION,
    ], cwd=PROJECT_ROOT)
else:
    print("Skipping enqueue step.")

print("Queue counts after enqueue step:")
print(queue_counts(QUEUE_URL))



$ 'c:\Users\26540\AppData\Local\Programs\Python\Python312\python.exe' 'C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\code\enqueue_ndvi_tasks.py' --queue-url https://sqs.us-east-1.amazonaws.com/891377197146/ndvi-province-tasks --tasks-csv 'C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\tasks_csv\province_tasks.csv' --bucket final-project-ndvi --manifest-prefix manifests --output-prefix outputs --script-key scripts/compute_hls_ndvi_by_province.py --boundary-key boundaries/geoBoundaries-CHN-ADM1.geojson --year 2018 --sqs-region us-east-1
Enqueued: Anhui -> anhui_ndvi_s3.txt
Enqueued: Beijing -> beijing_ndvi_s3.txt
Enqueued: Chongqing -> chongqing_ndvi_s3.txt
Enqueued: Fujian -> fujian_ndvi_s3.txt
Enqueued: Gansu -> gansu_ndvi_s3.txt
Enqueued: Guangdong -> guangdong_ndvi_s3.txt
Enqueued: Guangxi -> guangxi_ndvi_s3.txt
Enqueued: Guizhou -> guizhou_ndvi_s3.txt
Enqueued: Hebei -> hebei_ndvi_s3.txt
Enqueued: Heilongjia

## 9. Create or reuse EC2 key pair for SSH


In [16]:
ec2 = client("ec2", EC2_REGION)

def ensure_local_key_pair(key_name: str, pem_path: Path) -> Path:
    """
    Create/reuse an EC2 key pair.
    If the key pair already exists in AWS but the local .pem is missing,
    AWS cannot recover the private key. In that case, either use the correct .pem
    or choose a new KEY_NAME.
    """
    pem_path = Path(pem_path)
    pem_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        ec2.describe_key_pairs(KeyNames=[key_name])
        print(f"EC2 key pair already exists in {EC2_REGION}: {key_name}")
        if pem_path.exists():
            print("Local private key exists:", pem_path)
            return pem_path
        raise FileNotFoundError(
            f"Key pair {key_name} already exists in AWS, but local private key is missing: {pem_path}. "
            "AWS does not let you download an existing private key. Use the existing .pem file or change KEY_NAME."
        )
    except ClientError as e:
        code = e.response.get("Error", {}).get("Code")
        if code != "InvalidKeyPair.NotFound":
            raise

    print(f"Creating new EC2 key pair: {key_name}")
    response = ec2.create_key_pair(KeyName=key_name, KeyType="rsa")
    pem_path.write_text(response["KeyMaterial"], encoding="utf-8")

    if os.name == "nt":
        # Windows OpenSSH can reject keys with broad permissions. These commands usually fix that.
        run_local(["icacls", str(pem_path), "/inheritance:r"], check=False)
        run_local(["icacls", str(pem_path), "/grant:r", f"{os.environ.get('USERNAME', '%USERNAME%')}:(R)"], check=False)
    else:
        os.chmod(pem_path, 0o400)

    print("Created and saved private key:", pem_path)
    return pem_path

PEM_PATH = ensure_local_key_pair(KEY_NAME, PEM_PATH)


EC2 key pair already exists in us-west-2: ndvi-key
Local private key exists: C:\Users\26540\.ssh\ndvi-key.pem


## 10. Launch EC2 workers


In [18]:
LAUNCH_WORKERS = True

launch_cmd = [
    sys.executable,
    str(LAUNCHER_LOCAL),
    "--num-workers", str(NUM_WORKERS),
    "--region", EC2_REGION,
    "--sqs-region", SQS_REGION,
    "--queue-url", QUEUE_URL,
    "--project-bucket", PROJECT_BUCKET,
    "--worker-script-key", WORKER_SCRIPT_KEY,
    "--processing-script-key", PROCESSING_SCRIPT_KEY,
    "--cloud-processing-script-key", CLOUD_PROCESSING_SCRIPT_KEY,
    "--instance-type", INSTANCE_TYPE,
    "--volume-size-gb", str(VOLUME_SIZE_GB),
    "--swap-gb", str(SWAP_GB),
    "--iam-instance-profile", IAM_INSTANCE_PROFILE,
    "--security-group-name", SECURITY_GROUP_NAME,
    "--key-name", KEY_NAME,
    "--visibility-timeout", str(VISIBILITY_TIMEOUT),
    "--max-empty-polls", str(MAX_EMPTY_POLLS),
    "--output-csv", str(LAUNCHED_CSV),
]

if LAUNCH_WORKERS:
    run_local(launch_cmd, cwd=PROJECT_ROOT, timeout=1800)
else:
    print("Skipping launch step. Will read existing:", LAUNCHED_CSV)

if not LAUNCHED_CSV.exists():
    raise FileNotFoundError(f"Cannot find launched worker summary: {LAUNCHED_CSV}")

workers = pd.read_csv(LAUNCHED_CSV)
display(workers)



$ 'c:\Users\26540\AppData\Local\Programs\Python\Python312\python.exe' 'C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\code\launch_ndvi_workers.py' --num-workers 2 --region us-west-2 --sqs-region us-east-1 --queue-url https://sqs.us-east-1.amazonaws.com/891377197146/ndvi-province-tasks --project-bucket final-project-ndvi --worker-script-key scripts/worker_poll_sqs_ndvi_old.py --processing-script-key scripts/compute_hls_ndvi_by_province.py --cloud-processing-script-key scripts/compute_hls_ndvi_by_province_cloud_coverage.py --instance-type m5.large --volume-size-gb 100 --swap-gb 32 --iam-instance-profile LabInstanceProfile --security-group-name ndvi-worker-sg --key-name ndvi-key --visibility-timeout 21600 --max-empty-polls 5 --output-csv 'C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\code\launched_ndvi_workers.csv'
AWS account: 891377197146
Region: us-west-2
Default VPC: vpc-0b9748cfd33fc03d3
Selected subnet: subn

,instance_id,state,instance_type,private_ip,public_ip,public_dns,subnet_id,availability_zone
0,i-06371ad4a7292a4c4,running,m5.large,172.31.10.181,34.219.99.177,ec2-34-219-99-177.us-west-2.compute.amazonaws.com,subnet-0c33404dac71c5a77,us-west-2c
1,i-0d015a6b8269894dd,running,m5.large,172.31.3.246,54.202.169.172,ec2-54-202-169-172.us-west-2.compute.amazonaws...,subnet-0c33404dac71c5a77,us-west-2c


## 11. SSH commands for your workers


In [19]:
def make_ssh_command(ip: str) -> str:
    return f'ssh -i "{PEM_PATH}" ec2-user@{ip}'

ssh_table = workers.copy()
ssh_table["ssh_command"] = ssh_table["public_ip"].apply(make_ssh_command)
display(ssh_table[["instance_id", "state", "public_ip", "availability_zone", "ssh_command"]])


,instance_id,state,public_ip,availability_zone,ssh_command
0,i-06371ad4a7292a4c4,running,34.219.99.177,us-west-2c,"ssh -i ""C:\Users\26540\.ssh\ndvi-key.pem"" ec2-..."
1,i-0d015a6b8269894dd,running,54.202.169.172,us-west-2c,"ssh -i ""C:\Users\26540\.ssh\ndvi-key.pem"" ec2-..."


## 12. Connect to and prepare workers after launch

Run this section **after Section 11 gives worker public IPs**.

### 12.1 Local terminal: open the worker list and SSH into each worker

Run this in **PowerShell** from project root:
```
# Replace <PUBLIC_IP> with one public_ip from the CSV.
ssh -i "$env:USERPROFILE\.ssh\ndvi-key.pem" ec2-user@<PUBLIC_IP>
```

### 12.2 EC2 worker terminal: repair/setup the worker

After SSH into a worker, paste the full block below **on that EC2 worker**.

This block:
- checks the bootstrap log;
- keeps the existing `ndvi-venv` if it already works;
- creates `~/project`;
- downloads scripts from S3;
- recreates `start_worker.sh`;
- recreates the NASA credential template.

```bash
set -euxo pipefail

echo "===== Host ====="
hostname
date

echo "===== Disk before repair ====="
df -h /
lsblk

echo "===== Python environment check ====="
if [ ! -d /home/ec2-user/ndvi-venv ]; then
    sudo dnf update -y
    sudo dnf install -y python3 python3-pip python3-devel gcc gcc-c++ make awscli tmux
    python3 -m venv /home/ec2-user/ndvi-venv
    source /home/ec2-user/ndvi-venv/bin/activate
    python -m pip install --upgrade pip setuptools wheel
    pip install numpy pandas shapely geopandas rasterio boto3 awscli
else
    source /home/ec2-user/ndvi-venv/bin/activate
fi

python -c "import rasterio, geopandas, boto3, pandas, numpy; print('worker packages ok')"

echo "===== Create project folder and download scripts ====="
mkdir -p /home/ec2-user/project
cd /home/ec2-user/project

aws s3 cp s3://final-project-ndvi/scripts/worker_poll_sqs_ndvi_old.py /home/ec2-user/project/worker_poll_sqs_ndvi_old.py --no-progress --region us-west-2
aws s3 cp s3://final-project-ndvi/scripts/compute_hls_ndvi_by_province.py /home/ec2-user/project/compute_hls_ndvi_by_province.py --no-progress --region us-west-2
aws s3 cp s3://final-project-ndvi/scripts/compute_hls_ndvi_by_province_cloud_coverage.py /home/ec2-user/project/compute_hls_ndvi_by_province_cloud_coverage.py --no-progress --region us-west-2 || true

echo "===== Create start_worker.sh ====="
cat > /home/ec2-user/project/start_worker.sh <<'STARTWORKER'
#!/usr/bin/env bash
set -euo pipefail

source /home/ec2-user/ndvi-venv/bin/activate

python3 /home/ec2-user/project/worker_poll_sqs_ndvi_old.py \
  --queue-url "https://sqs.us-east-1.amazonaws.com/891377197146/ndvi-province-tasks" \
  --sqs-region us-east-1 \
  --nasa-profile nasa \
  --visibility-timeout 21600 \
  --max-empty-polls 5
STARTWORKER

chmod +x /home/ec2-user/project/start_worker.sh

echo "===== Create NASA credential refresh template ====="
cat > /home/ec2-user/project/refresh_nasa_credentials_template.sh <<'REFRESH'
#!/usr/bin/env bash
set +o history

aws configure set aws_access_key_id "PASTE_NEW_ACCESS_KEY" --profile nasa
aws configure set aws_secret_access_key "PASTE_NEW_SECRET_KEY" --profile nasa
aws configure set aws_session_token "PASTE_NEW_SESSION_TOKEN" --profile nasa
aws configure set region us-west-2 --profile nasa

set -o history
aws configure list --profile nasa
REFRESH

chmod +x /home/ec2-user/project/refresh_nasa_credentials_template.sh
sudo chown -R ec2-user:ec2-user /home/ec2-user/project /home/ec2-user/ndvi-venv

echo "===== Final check ====="
ls -lh /home/ec2-user/project
df -h /
```

Run the same block on every worker.


## 13. Configure NASA temporary credentials on each worker

Get fresh **NASA Earthdata S3 temporary credentials** first. Then run these commands **inside each EC2 worker**.

### 13.1 Edit the template

```bash
cd ~/project
nano refresh_nasa_credentials_template.sh
```

Replace:

```text
PASTE_NEW_ACCESS_KEY
PASTE_NEW_SECRET_KEY
PASTE_NEW_SESSION_TOKEN
```

with fresh NASA credentials.

Then run:

```bash
bash ~/project/refresh_nasa_credentials_template.sh
```

### 13.2 Alternative: configure directly without editing a file

Run this **inside each EC2 worker**:

```bash
aws configure set aws_access_key_id "YOUR_NASA_ACCESS_KEY" --profile nasa
aws configure set aws_secret_access_key "YOUR_NASA_SECRET_KEY" --profile nasa
aws configure set aws_session_token "YOUR_NASA_SESSION_TOKEN" --profile nasa
aws configure set region us-west-2 --profile nasa

aws configure list --profile nasa
```


## 14. Start NDVI workers in tmux

Run this **inside each EC2 worker** after the NASA profile is configured.

### 14.1 Start worker in a detached tmux session

```bash
cd ~/project

# Start a new worker session in the background.
tmux new-session -d -s ndvi 'cd ~/project && ./start_worker.sh 2>&1 | tee -a ~/project/worker.log'

# Check that it started.
tmux ls
tail -n 50 ~/project/worker.log
```

### 14.2 Monitor the worker

```bash
tmux attach -t ndvi
```

Detach without stopping the worker:

```text
Ctrl + B, then D
```


## 15. Monitor SQS progress

Run this from **local PowerShell terminal**, not inside EC2.

```powershell
aws sqs get-queue-attributes `
  --queue-url "https://sqs.us-east-1.amazonaws.com/891377197146/ndvi-province-tasks" `
  --attribute-names ApproximateNumberOfMessages ApproximateNumberOfMessagesNotVisible ApproximateNumberOfMessagesDelayed `
  --region us-east-1
```

Interpretation:

```text
ApproximateNumberOfMessages             = tasks waiting in queue
ApproximateNumberOfMessagesNotVisible   = tasks currently being processed by workers
ApproximateNumberOfMessagesDelayed      = delayed tasks
```

When both `ApproximateNumberOfMessages` and `ApproximateNumberOfMessagesNotVisible` are `0`, the queue is probably finished.


## 17. Download NDVI outputs from S3 

Run this from **local PowerShell terminal** after the queue is drained or after some provinces have completed.

```powershell
cd "C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness"

New-Item -ItemType Directory -Force ".\ndvi_output\s3_outputs" | Out-Null

aws s3 sync `
  "s3://final-project-ndvi/outputs/" `
  ".\ndvi_output\s3_outputs\" `
  --region us-west-2
```

Then come back to this notebook and continue with **Section 18** to combine province annual/daily CSV files.



## Optional cleanup: terminate workers

Only run this after outputs are safely downloaded and you no longer need the workers.


In [ ]:
TERMINATE_WORKERS = False

if TERMINATE_WORKERS:
    instance_ids = workers["instance_id"].dropna().astype(str).tolist()
    if not instance_ids:
        raise RuntimeError("No instance IDs found.")
    print("Terminating:", instance_ids)
    ec2.terminate_instances(InstanceIds=instance_ids)
    print("Termination requested.")
else:
    print("Skipping worker termination.")
